# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [6]:
#pip  install transformers sentencepiece torch


In [7]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [8]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
c:\Users\jaber\miniconda3\Lib\site-packages\transformers\generation\utils.py:1955: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [10]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

### Answer

| # | Task | Model family | Why |
|---|------|--------------|-----|
| 1 | Sentiment classification | **Encoder-only** (BERT, RoBERTa) | Reads the whole sentence in both directions and outputs one label. |
| 2 | Machine translation | **Encoder–decoder** (T5, BART) | The encoder reads the source sentence and the decoder writes a new sentence in another language. |
| 3 | Open-ended text generation | **Decoder-only** (GPT family) | Predicts the next token from the previous ones, so it can keep writing text. |
| 4 | Named Entity Recognition | **Encoder-only** (BERT, RoBERTa) | Gives a label to every token, which needs the full context on both sides. |
| 5 | Text summarization | **Encoder–decoder** (T5, BART) | Reads a long input and writes a shorter, new output text. |


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [11]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

results = classifier(sentences)

for sentence, result in zip(sentences, results):
    print(f"{sentence:<42} -> {result['label']:<8} (score: {result['score']:.4f})")


The movie was excellent.                   -> POSITIVE (score: 0.9999)
The lecture was difficult to understand.   -> NEGATIVE (score: 0.9997)
I enjoyed learning about transformers.     -> POSITIVE (score: 0.9997)


## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [12]:
paragraph = (
    "Natural language processing is a field of artificial intelligence that helps computers "
    "understand human language. It is used in many applications such as machine translation, "
    "chatbots, spam filtering, and sentiment analysis. Modern NLP systems are based on transformer "
    "models, which are trained on very large amounts of text and can learn the meaning of words "
    "from their context."
)

# T5 was trained with the lowercase "summarize:" prefix, which gives the cleanest summary
prompt = "summarize: " + paragraph

inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=60)

print("Summary:", tokenizer.decode(output[0], skip_special_tokens=True))


Summary: Using natural language processing, computers can learn the meaning of words from their context.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


### Answer

A decoder-only model is trained to predict the next token from all the tokens before it, which is exactly what a chatbot does when it writes a reply word by word. The conversation so far (the user's message and the earlier turns) is simply given as the prompt, and the model continues it with an answer, so there is no need for a separate encoder. Because it is trained on huge amounts of text, it learns grammar, facts, and how to follow instructions, and it can produce long, open-ended answers instead of only a fixed label. Its causal (left-to-right) attention also makes generation efficient, since earlier tokens do not have to be recomputed for every new token.


# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [13]:
prompt = "Natural language processing is"

# Sampling without a fixed seed: each run picks different next tokens
print("Without set_seed:")
for i in range(3):
    output = generator(prompt, max_new_tokens=30, do_sample=True, top_k=50, temperature=0.9)
    print(f"Run {i + 1}: {output[0]['generated_text']}")
    print()

# Sampling with set_seed(42) before each run: the output is the same every time
print("With set_seed(42):")
for i in range(2):
    set_seed(42)
    output = generator(prompt, max_new_tokens=30, do_sample=True, top_k=50, temperature=0.9)
    print(f"Run {i + 1}: {output[0]['generated_text']}")
    print()


[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'top_k', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Without set_seed:


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 1: Natural language processing is currently a topic of debate. Language processing is one of the most effective tools currently used by the government. Many people consider it the safest approach to processing



[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 2: Natural language processing is key in any language processing program and does not require any other technical support. For more information or to support the application in writing, please contact us at



[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 3: Natural language processing is usually a sign of the beginning of a new age, but it could turn out that the language processing process will have an impact on the quality of the

With set_seed(42):


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 1: Natural language processing is more prevalent worldwide than is used in traditional languages, because it is harder to understand and understand these languages. The research team found that most languages are languages

Run 2: Natural language processing is more prevalent worldwide than is used in traditional languages, because it is harder to understand and understand these languages. The research team found that most languages are languages



### Answer

With `do_sample=True` the model does not always choose the most likely next token. It computes a probability for every token in the vocabulary and then **randomly samples** one of them (here from the 50 most likely tokens, `top_k=50`). Because a different token can be picked at any step, and every later token depends on the earlier ones, small differences grow into completely different texts. `set_seed(42)` fixes the random number generator, so the same "random" choices are made each time and the output can be reproduced. Greedy decoding (`do_sample=False`, as in Part A) always gives the same output because it always picks the highest-probability token.
